In [ ]:
import importlib
import numpy as np
import h5py
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, accuracy_score
import classification
importlib.reload(classification)

**Time series data**

In [ ]:
# Basic necessary functions

# Create masks to select certain subsystems
def subsys(channel_names):
    subsys_counts = {}
    subsys_masks = {}
    n = len(channel_names)
    for i, name in enumerate(channel_names):
        key = name[3:6]

        subsys_counts[key] = subsys_counts.get(key, 0) + 1

        if key not in subsys_masks:
            subsys_masks[key] = np.zeros(n, dtype=bool)
        subsys_masks[key][i] = True
    
    return subsys_masks, subsys_counts

# Create mask for to select only samples of certain classes
def class_mask(labels):
    n = labels == 0
    t = labels == 1
    w = labels == 2
    s = labels == 3
    
    return n,t,w,s

# Remap the labels such that you always get succesive labels starting from zero. (0,1) instead of (1,3)
def remap_labels(labels):
    unique = np.unique(labels)
    mapping = {old: new for new, old in enumerate(unique)}
    remapped = np.vectorize(mapping.get)(labels)
    return remapped, mapping

In [ ]:
# Open a .txt file containing the channel names
with open("channel_names.txt", "r") as f:
    channel_names = [line.strip() for line in f]
subsys_masks, subsys_counts = subsys(channel_names)


# Load data  -- This is raw time series data of shape [N_samples, N_timesteps, N_channels] which in our case is [N_samples=2561, 2048, 783]
data = np.load('combined_data_NTWS.npy')        # NTWS tells the order in which different classes are present: N = normal, T = Tomte, W = Whistle, S = Scattered Light
labels = np.load('combined_labels_NTWS.npy')

n,t,w,s = class_mask(labels)

In [ ]:
# How to run the model
classes = (n,s)
mask_class = np.logical_or.reduce(classes)
remapped_labs, mapping = remap_labels(labels[mask_class])

subsys_to_use = ('SUS', 'IMC')
subsys_to_use_masks = [subsys_masks[sys] for sys in subsys_to_use]
mask_subsys = np.logical_or.reduce(subsys_to_use_masks)

data_flipped = np.transpose(data[:,:,:], (0,2,1))
# If you want to use a subset of channels:
# data_flipped = data_flipped[:,:,mask_subys]

print(data_flipped.shape)
print(f'Number of nodes: {data_flipped.shape[1]}')
accuracy, (true_labels, pred_labels), (train_loss_list, val_loss_list), lr_list, (val_acc_list, train_acc_list), importance_metrics = classification.GAT_classifier(data=data_flipped[mask_class,:,:], labels=remapped_labs, 
               balance_classes= True, dropout=0.255,
               n_epochs = 750, lr = 5e-5,  topk = 12, similarity_type = 'cosine', mode = 'mean', 
               num_layers = 1, pool_type = 'mean',
               seed = 15, save_imgs = False, patience=108,
               T_max = 750, eta_min = 5e-8, batch_size = 32, train_frac=0.7, val_frac = 0.15, preprocess=True)

In [ ]:
# Visualise Training

epochs = np.arange(1, len(train_loss_list) + 1)

fig, ax1 = plt.subplots()

# Possibly also plot train/val accuracy on the right y-axis
# Left y-axis: Loss
ax1.plot(epochs, train_loss_list, label='Train Loss')
ax1.plot(epochs, val_loss_list, label='Val Loss')
#ax1.set_yscale('log')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend(loc='upper left')
#ax1.grid()
# Right y-axis: Accuracy
#ax2 = ax1.twinx()
#ax2.plot(epochs, train_acc_list, linestyle='--', label='Train acc')
#ax2.plot(epochs, val_acc_list, linestyle='--', label='Val acc')

#ax2.set_ylabel('Accuracy')

#ax2.legend(loc='upper right')

#plt.title('Training and Validation Loss and Accuracy')
plt.show()
plt.close()


cm = confusion_matrix(true_labels, pred_labels)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt="", cmap="Blues", cbar=False,
                xticklabels=('Normal', 'Scattered Light'), yticklabels=('Normal', 'Scattered Light'))
plt.xlabel("Predicted label")
plt.ylabel("True label")
#plt.title("GAT Confusion Matrix")
plt.show()
plt.close()


print(f'Accuracy: {accuracy_score(true_labels, pred_labels):.2%}')

**Calculating channel importance**
Using channels as nodes

In [ ]:
# Function for calculating outgoing attention per node
def outgoing_attention_per_node(edge_idx, att_w, num_nodes):
    """
    edge_idx: (2, E)
    att_w: (E,) or (E,1)
    returns: (num_nodes,)
    """

    edge_idx = np.asarray(edge_idx)
    att_w = np.asarray(att_w).squeeze()  # (E,)

    src = edge_idx[0]  # outgoing nodes
    # tgt = edge_idx[1]  # not needed here

    node_sum = np.zeros(num_nodes)
    node_count = np.zeros(num_nodes)

    np.add.at(node_sum, src, att_w)
    np.add.at(node_count, src, 1)

    # avoid division by zero
    node_avg = np.divide(
        node_sum,
        node_count,
        out=np.zeros_like(node_sum),
        where=node_count != 0
    )

    return node_avg

In [ ]:
# Generate mask for normal and glitch data
normal_samps_mask = (np.array(true_labels)==0)
glitch_samps_mask = (true_labels==1)

# Use mask to get attention weights and edge indices per class
normal_attweights = np.array(importance_metrics[1])[normal_samps_mask]
normal_edge_indices = np.array(importance_metrics[0])[normal_samps_mask]

glitch_attweights = np.array(importance_metrics[1])[glitch_samps_mask]
glitch_edge_indices = np.array(importance_metrics[0])[glitch_samps_mask]

In [ ]:
# Calculate the attention per node
num_nodes = 783

normal_node_att = np.array([
    outgoing_attention_per_node(
        normal_edge_indices[i],
        normal_attweights[i],
        num_nodes
    )
    for i in range(len(normal_attweights))
])

glitch_node_att = np.array([
    outgoing_attention_per_node(
        glitch_edge_indices[i],
        glitch_attweights[i],
        num_nodes
    )
    for i in range(len(glitch_attweights))
])

# Take mean across all samples
normal_class_attention = normal_node_att.mean(axis=0)
glitch_class_attention = glitch_node_att.mean(axis=0)

In [ ]:
# Calculate the difference between attention weights across classes
delta_attention = np.abs(normal_class_attention - glitch_class_attention)
important_idxs = np.argsort(delta_attention)[::-1]
print(np.array(channel_names)[important_idxs[:10]])

In [ ]:
# Function for checking the important witness channels of scattered light glitch according to our classifier model
def get_channel_ranks(channels, df):
    return (
        df[df["channel"].isin(channels)]
        .loc[:, ["channel",
                 "delta_attention", "delta_rank"
                 ]]
        .sort_values("delta_rank")
    )
channels_to_check = ["L1:SUS-BS_M2_WIT_L_DQ", "L1:SUS-ETMX_L2_WIT_L_DQ", "L1:SUS-ETMX_M0_DAMP_L_IN1_DQ", "L1:SUS-ETMY_L2_WIT_L_DQ",
                     "L1:SUS-ETMY_M0_DAMP_L_IN1_DQ", "L1:SUS-OM1_M1_DAMP_L_IN1_DQ", "L1:SUS-OM3_M1_DAMP_L_IN1_DQ", "L1:SUS-SR2_M1_DAMP_L_IN1_DQ",
                     "L1:SUS-SR2_M3_WIT_L_DQ", "L1:SUS-BS_M1_DAMP_L_IN1_DQ", "L1:SUS-OM2_M1_DAMP_L_IN1_DQ", "L1:SUS-PR2_M1_DAMP_L_IN1_DQ", 
                     "L1:SUS-PR2_M3_WIT_L_DQ", "L1:SUS-RM2_M1_DAMP_L_IN1_DQ"
                     ]
get_channel_ranks(channels_to_check, df)

**FD data**

In [ ]:
# Open data
full_clean = h5py.File("/data/gravwav/lopezm/Projects/AnomalyDetection/anushka/frames/stacked_sub/sub_Clean_L1_O3a.hdf5", mode = 'r')
full_scat = h5py.File("/data/gravwav/lopezm/Projects/AnomalyDetection/anushka/frames/stacked_sub/sub_Scattered_Light_L1_O3a.hdf5", mode = 'r')
full_whistle = h5py.File("/data/gravwav/lopezm/Projects/AnomalyDetection/anushka/frames/stacked_sub/sub_Whistle_L1_O3a.hdf5", mode = 'r')
full_tomte = h5py.File("/data/gravwav/lopezm/Projects/AnomalyDetection/anushka/frames/stacked_sub/sub_Tomte_L1_O3a.hdf5", mode = 'r')

full_data_clean = full_clean['submatrices'][:]
full_data_scat = full_scat['submatrices'][:]
full_data_whistle = full_whistle['submatrices'][:]
full_data_tomte = full_tomte['submatrices'][:]

# Preprocessing functions
def minmax_normalize(x):
    # Helper function to minmax scale the data
    min_val = x.min(axis=(0,2), keepdims=True)
    max_val = x.max(axis=(0,2), keepdims=True)
    return (x-min_val) / (max_val - min_val + 1e-8)

def norm_take_out_nans(x):
    # Take out all channels containing NaN's and normalize data (between 0-1)
    # Convert bytes -> str
    channels = np.array([ch.decode('utf-8') for ch in full_clean['channels']])
    # Boolean mask: True where channel is in channel_names
    mask = np.isin(channels, channel_names)
    no_NaN0 = x[:,mask,:]

    ch2 = channels[mask]

    mask2 = ~np.isnan(no_NaN0).any(axis=(0,2))
    no_NaN = no_NaN0[:,mask2,:]

    ch3 = ch2[mask2]
    # Normalize data
    first_window_norm = minmax_normalize(no_NaN[:,:,0:32])
    second_window_norm = minmax_normalize(no_NaN[:,:,32:48])
    third_window_norm = minmax_normalize(no_NaN[:,:,48:56])
    fourth_window_norm = minmax_normalize(no_NaN[:,:,56:])
    return np.concatenate((first_window_norm, second_window_norm, third_window_norm, fourth_window_norm), axis=2), ch3


In [ ]:
# Loading data, preprocessing it and generating channel names and labels
FD_data = np.concatenate((full_data_clean, full_data_tomte, full_data_whistle, full_data_scat), axis=0)
FD_data_norm, ch_FD = norm_take_out_nans(FD_data)
FD_labels = np.concatenate((np.zeros(full_data_clean.shape[0]), np.ones(full_data_tomte.shape[0]), 2*np.ones(full_data_whistle.shape[0]), 3*np.ones(full_data_scat.shape[0])), axis=0)

# FD_data_norm has shape: [N_samples, N_channels, N_steps]  NOTE this is different compared to time series data
# More specifically FD_data_norm : [N_samples=42011, 769, 60]

In [ ]:
# Generate masks for classes and channel subsystems
subsys_masks_FD, subsys_counts_FD = subsys(ch_FD)
n_FD,t_FD,w_FD,s_FD = class_mask(FD_labels)

classes_FD = (n_FD, s_FD)
mask_class_FD = np.logical_or.reduce(classes_FD)
remapped_labs_FD, mapping_FD = remap_labels(FD_labels[mask_class_FD])

subsys_to_use_FD = ('SUS',)
subsys_to_use_masks_FD = [subsys_masks_FD[sys] for sys in subsys_to_use_FD]
mask_subsys_FD = np.logical_or.reduce(subsys_to_use_masks_FD)

In [ ]:
# Train model

FD_flipped = np.transpose(FD_data_norm, (0,2,1))

# If you want to use a subset of channels:
# FD_flipped = FD_flipped[:,mask_subys,:]

print(FD_flipped.shape)
print(f'Number of nodes: {FD_flipped.shape[1]}')

print(f'Number of nodes: {FD_flipped.shape[1]}')
accuracy_FD, (true_labels_FD, pred_labels_FD), (train_loss_list_FD, val_loss_list_FD), lr_list_FD, (val_acc_list_FD, train_acc_list_FD), importance_metrics_FD = classification.GAT_classifier(
    data=FD_flipped[mask_class_FD,:,:], labels=remapped_labs_FD, balance_classes= True, 
               n_epochs = 500, lr = 5e-5,  dropout = 0.518, 
               topk = 10, similarity_type = 'pearson', mode = 'mean', 
               num_layers = 1, pool_type = 'mean',
               seed = 18, save_imgs = False, patience=65, preprocess=True,
               T_max = 500, eta_min = 5e-9, batch_size = 32, train_frac=0.7, val_frac=0.15)


In [ ]:
# Visualise training

epochs_FD = np.arange(1, len(train_loss_list_FD) + 1)

fig, ax1 = plt.subplots()

# Left y-axis: Loss
ax1.plot(epochs_FD, train_loss_list_FD, label='Train Loss')
ax1.plot(epochs_FD, val_loss_list_FD, label='Val Loss')
#ax1.set_yscale('log')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend(loc='upper left')


# Possibly also plot train/val accuracy on the right y-axis
# Right y-axis: Accuracy
#ax2 = ax1.twinx()
#ax2.plot(epochs_FD, train_acc_list_FD, linestyle='--', label='Train acc')
#ax2.plot(epochs_FD, val_acc_list_FD, linestyle='--', label='Val acc')
#ax2.set_ylabel('Accuracy')
#ax2.legend(loc='upper right')

#plt.title('Training and Validation Loss and Accuracy')
plt.show()
plt.close()


cm = confusion_matrix(true_labels_FD, pred_labels_FD)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt="", cmap="Blues", cbar=False,
                xticklabels=('Normal', 'Scattered Light'), yticklabels=('Normal', 'Scattered Light'))
plt.xlabel("Predicted label")
plt.ylabel("True label")
#plt.title("GAT Confusion Matrix")
plt.show()
plt.close()


print(f'Accuracy: {accuracy_score(true_labels_FD, pred_labels_FD):.2%}')

**Calculating channel importance**
Using time dimension as nodes

In [ ]:
# Function for calculating channel importance using the weight matrix 
def importance_per_channel_FD(metric):
    metric_np = metric.detach().cpu().numpy()

    # Sum absolute weights over the output dimensions
    importance = np.sum(np.abs(metric_np), axis=0)

    # Indices sorted from lowest to highest importance
    important_ids = np.argsort(importance)

    return importance, important_ids


# Compute importance scores
importance_FD, important_ids_FD = importance_per_channel_FD(importance_metrics_FD[4])

# Get the 10 most important channels
top10_ids_FD = important_ids_FD[-20:][::-1]

print("Top 20 most important channels:")
for rank, idx in enumerate(top10_ids_FD, start=1):
    print(f"{rank:2d}. {ch_FD[idx]:<30} {importance_FD[idx]:.4f}")

In [ ]:
# Function for checking the important witness channels of scattered light glitch according to our classifier model based on time steps as nodes

def print_channel_ranks(channels_to_check, ch_all, importance):
    """
    Prints rank + importance for selected channels.
    """

    # Ensure numpy array
    ch_all = np.array(ch_all)

    # Sort indices (high -> low importance)
    sorted_ids = np.argsort(importance)[::-1]

    # Build rank lookup: index -> rank
    rank_lookup = {idx: rank + 1 for rank, idx in enumerate(sorted_ids)}

    # Build name -> index mapping (FIX)
    name_to_idx = {name: i for i, name in enumerate(ch_all)}

    print("Channel rankings (1 = most important):\n")

    for name in channels_to_check:
        name = "L1:" + name
        if name not in name_to_idx:
            print(f"{name:45s} -> NOT FOUND")
            continue

        idx = name_to_idx[name]
        rank = rank_lookup[idx]
        score = importance[idx]

        print(f"{name:45s} -> rank {rank:4d}, importance {score:.6f}")


channels_to_check = ["SUS-BS_M2_WIT_L_DQ", "SUS-ETMX_L2_WIT_L_DQ", "SUS-ETMX_M0_DAMP_L_IN1_DQ", "SUS-ETMY_L2_WIT_L_DQ",
                     "SUS-ETMY_M0_DAMP_L_IN1_DQ", "SUS-OM1_M1_DAMP_L_IN1_DQ", "SUS-OM3_M1_DAMP_L_IN1_DQ", "SUS-SR2_M1_DAMP_L_IN1_DQ",
                     "SUS-SR2_M3_WIT_L_DQ", "SUS-BS_M1_DAMP_L_IN1_DQ", "SUS-OM2_M1_DAMP_L_IN1_DQ", "SUS-PR2_M1_DAMP_L_IN1_DQ", 
                     "SUS-PR2_M3_WIT_L_DQ", "SUS-RM2_M1_DAMP_L_IN1_DQ"
                     ]
print_channel_ranks(channels_to_check, ch_FD, importance_FD)